### np.log10

In [1]:
import math

def get_shape(a):
    if not isinstance(a, list):
        return ()
    if len(a) == 0:
        return (0,)
    inner_shape = get_shape(a[0])
    for i in range(1, len(a)):
        if get_shape(a[i]) != inner_shape:
            raise ValueError(
                f"Jagged array: first row shape {inner_shape}, "
                f"but row {i} shape {get_shape(a[i])}"
            )
    return (len(a),) + inner_shape


def _build_nested(shape, getter, prefix=()):
    if len(shape) == 0:
        return getter(prefix)
    return [_build_nested(shape[1:], getter, prefix + (i,)) for i in range(shape[0])]


def _get_item(a, idx):
    cur = a
    for i in idx:
        cur = cur[i]
    return cur


def _log10_scalar(x):
    if x > 0:
        return math.log10(x)
    if x == 0:
        return float('-inf')
    return float('nan')


def np_log10(a):
    shape = get_shape(a)
    if len(shape) == 0:
        return _log10_scalar(a)

    def getter(idx):
        return _log10_scalar(_get_item(a, idx))

    return _build_nested(shape, getter)

### test cases 

In [2]:
print(np_log10(1))
print(np_log10(10))
print(np_log10(100))
print(np_log10(1000))
print(np_log10([1, 10, 100, 1000, 10000]))
print(np_log10([[1, 10], [100, 1000]]))
print(np_log10([[[1, 10], [100, 1000]], [[10000, 100000], [1000000, 10000000]]]))
print(np_log10(0))
print(np_log10(-1))
print(np_log10([-1, 0, 1, 10]))

0.0
1.0
2.0
3.0
[0.0, 1.0, 2.0, 3.0, 4.0]
[[0.0, 1.0], [2.0, 3.0]]
[[[0.0, 1.0], [2.0, 3.0]], [[4.0, 5.0], [6.0, 7.0]]]
-inf
nan
[nan, -inf, 0.0, 1.0]


In [3]:
print(np_log10([[1, 2], [3]]))

ValueError: Jagged array: first row shape (2,), but row 1 shape (1,)

In [4]:
print(np_log10([1, "a"]))

TypeError: '>' not supported between instances of 'str' and 'int'

In [5]:
print(np_log10(1 + 1j))

TypeError: '>' not supported between instances of 'complex' and 'int'